In [1]:
import numpy as np
import pandas as pd
import warnings
from datetime import date
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from models import FeedForward, LSTM, Hybrid

warnings.filterwarnings('ignore')

In [2]:
import os
from torch.utils.data import Dataset

def compute_valid_indices(tickers: np.ndarray, seq_len: int) -> np.ndarray:
    """Indices i where window [i-seq_len:i) stays within same ticker segment."""
    t = np.asarray(tickers)
    n = len(t)
    idx = []
    start = 0
    for i in range(1, n + 1):
        if i == n or t[i] != t[i - 1]:
            end = i
            if end - start > seq_len:
                idx.extend(range(start + seq_len, end))
            start = end
    return np.asarray(idx, dtype=np.int32)

class SequenceDataset(Dataset):
    """Lazily yields sliding windows without materializing X_seq in RAM."""
    def __init__(self, X: np.ndarray, y: np.ndarray, tickers: np.ndarray, seq_len: int, to_dtype=torch.float32):
        # ✅ FIX: Keep tensors on CPU initially
        self.X = torch.from_numpy(X.astype(np.float32, copy=False))
        self.y = np.asarray(y, dtype=np.int8, order='C')  # smaller; convert to long on fetch
        self.seq_len = seq_len
        self.valid_idx = compute_valid_indices(tickers, seq_len)
        self.to_dtype = to_dtype

    def __len__(self):
        return len(self.valid_idx)

    def __getitem__(self, i):
        idx = self.valid_idx[i]
        x = self.X[idx - self.seq_len: idx].to(dtype=self.to_dtype)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        return x, y
    
class CUDAPrefetchLoader:
    def __init__(self, loader, device, dtype, prefetch=4):
        self.loader, self.device, self.dtype, self.prefetch = loader, device, dtype, prefetch
        self.stream = torch.cuda.Stream() if device.type == 'cuda' else None

    def __iter__(self):
        if self.stream is None:
            yield from self.loader
            return

        it, cache = iter(self.loader), []
        with torch.cuda.stream(self.stream):
            for _ in range(self.prefetch):
                try:
                    bx, by = next(it)
                except StopIteration:
                    break
                cache.append((
                    bx.to(self.device, dtype=self.dtype, non_blocking=True),
                    by.to(self.device, dtype=self.dtype, non_blocking=True)
                ))

        while cache:
            torch.cuda.current_stream().wait_stream(self.stream)
            batch = cache.pop(0)

            try:
                nx, ny = next(it)
                with torch.cuda.stream(self.stream):
                    cache.append((
                        nx.to(self.device, dtype=self.dtype, non_blocking=True),
                        ny.to(self.device, dtype=self.dtype, non_blocking=True)
                    ))
            except StopIteration:
                pass
            yield batch
            
    def __len__(self):
        return len(self.loader)

In [3]:
# READ S&P 500 TICKERS FROM TXT
def load_simple_ticker_list(filename='sp500_tickers.txt'):
    """Load tickers from simple text file (one per line)"""
    try:
        with open(filename, 'r') as f:
            tickers = [line.strip() for line in f.readlines() if line.strip()]
        return tickers
    except FileNotFoundError:
        print(f"❌ File {filename} not found.")
        return []

tickers = load_simple_ticker_list('sp500_tickers.txt')

In [4]:
# S&P 500 STOCK PROCESSING WITH LIMITS
import gc  # For garbage collection

# STOCK LIMIT CONFIGURATION
USE_FULL_DATASET = True  # Set to True when ready for full run
STOCK_LIMIT = 5          # Limit for testing (set to desired number)

# Apply stock limit
if USE_FULL_DATASET:
    processing_tickers = tickers
    print(f"🚀 FULL DATASET MODE: Processing all {len(processing_tickers)} S&P 500 stocks")
else:
    processing_tickers = tickers[:STOCK_LIMIT]
    print(f"🧪 TEST MODE: Processing first {len(processing_tickers)} stocks")

print("=" * 60)
print(f"📊 Dataset: S&P 500")
print(f"🎯 Stocks to process: {len(processing_tickers)}")
print("=" * 60)

ticker_to_code = {t: i for i, t in enumerate(processing_tickers)}

all_data = []
total_tickers = len(processing_tickers)

# For progress tracking
successful_stocks = 0
failed_stocks = 0

for idx, ticker in enumerate(processing_tickers, 1):
    try:
        features = pd.read_parquet(f'processed_features_cleaned2/{ticker}.parquet')
        features['Ticker'] = np.int16(ticker_to_code[ticker])
        
        float_cols = features.select_dtypes(include=['float64', 'float32']).columns
        features[float_cols] = features[float_cols].astype(np.float32, copy=False)

        returns = features['Close'].pct_change(periods=4).shift(-4)
        
        # Method 1: Use percentiles for balanced classes
        q33 = returns.quantile(0.33)
        q67 = returns.quantile(0.67)
        
        features['Target'] = np.where(returns > q67, 2, np.where(returns < q33, 0, 1)).astype(np.int8, copy=False)
        
        all_data.append(features)
        
        # GARBAGE COLLECTION
        if idx % 25 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"\r🧹 [{idx}/{total_tickers}] Memory cleanup completed" + " " * 30)
            
    except Exception as e:
        print(f"\r❌ [{idx}/{total_tickers}] {ticker}: Error - {str(e)}")
        continue

# Final cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Combine all data
if all_data:
    ml_data = pd.concat(all_data, ignore_index=True)
    
    print(f"📊 Total samples: {len(ml_data):,}")
    print(f"📈 Stocks: {len(ml_data['Ticker'].unique())}")
    print(f"💾 Memory: {ml_data.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
else:
    print("\n❌ ERROR: No data processed successfully!")
    ml_data = pd.DataFrame()
    
del all_data
gc.collect()

🚀 FULL DATASET MODE: Processing all 503 S&P 500 stocks
📊 Dataset: S&P 500
🎯 Stocks to process: 503


KeyboardInterrupt: 

In [ ]:
#Device Detection
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon GPU")
else:
    device = torch.device('cpu')
    print("Using CPU")

In [ ]:
# DATA PREPARATION FOR PYTORCH
print("📊 PREPARING DATA FOR PYTORCH")
print("=" * 40)

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from typing import Any

def prep_training_data(ml_data, sequence_length=16, test_size=0.2, use_bf16=True) -> dict[str, Any]:
    """
    Prepare data for PyTorch training
    Args:
        ml_data (pd.DataFrame): Your dataset
        sequence_length (int): Number of days to look back (for LSTM)
        test_size (int): Fraction for testing
    Returns:
        data (dict): With torch float sensors sent to the {device}
    """
    # Separate features and target
    feature_cols = [col for col in ml_data.columns if col not in ['Target', 'Ticker']]    
    ml_data.dropna(inplace=True)
        
    print(f"📋 Input data shape: {ml_data.shape}")
    
    # --- Split data using train_test_split WITHOUT shuffling ---
    X_train, X_test, y_train, y_test, t_train, t_test = train_test_split(
        ml_data[feature_cols], 
        ml_data['Target'],
        ml_data['Ticker'],
        test_size=test_size,
        shuffle=False # Don't shuffle time-series data
    )
    
    del ml_data
    gc.collect()

    # --- FIX: Fit scaler ONLY on training data ---
    feature_scaler = RobustScaler()
    X_train = feature_scaler.fit_transform(X_train).astype(np.float32, copy=False)
    X_test = feature_scaler.transform(X_test).astype(np.float32, copy=False)
    
    # Convert to appropriate dtype based on BF16 setting
    if use_bf16:
        print("🔥 Using BF16 precision for features")
        tensor_dtype = torch.bfloat16
    else:
        print("📊 Using FP32 precision for features")
        tensor_dtype = torch.float32
    
    # Handle target scaling correctly
    y_train = y_train.to_numpy(dtype=np.int8, copy=False)
    y_test = y_test.to_numpy(dtype=np.int8, copy=False)
    t_train = t_train.to_numpy(dtype=np.int16, copy=False)
    t_test  = t_test.to_numpy(dtype=np.int16, copy=False)
    
    print(f"💾 Memory after conversion: {(X_train.nbytes + X_test.nbytes + y_train.nbytes + y_test.nbytes + t_train.nbytes + t_test.nbytes) / 1024**2:.1f} MB")

    # Convert to PyTorch tensors
    data = {
        # 'regular': {
        #     'X_train': torch.from_numpy(X_train).to(dtype=tensor_dtype),
        #     'X_test': torch.from_numpy(X_test).to(dtype=tensor_dtype),
        #     'y_train': torch.from_numpy(y_train),
        #     'y_test': torch.from_numpy(y_test)
        # },
        # 'sequence': {
        #     'X_train': None,  # will be built lazily by Dataset
        #     'X_test': None,
        #     'y_train': None,
        #     'y_test': None
        # },
        'raw': {  # needed to build SequenceDataset lazily
            'X_train': X_train, 'y_train': y_train, 't_train': t_train,
            'X_test': X_test,   'y_test': y_test,   't_test': t_test
        },
        'feature_names': feature_cols,
        'sequence_length': sequence_length,
        'use_bf16': use_bf16,
        'tensor_dtype': tensor_dtype,
        "predicting_returns": True
    }
    
    print(f"✅ Regular data - Train: {X_train.shape}, Test: {X_test.shape}")
    print(f"✅ Tensor dtype: {tensor_dtype}")
    print(f"✅ Sequence data - built lazily in DataLoader")
    print(f"📊 Features: {len(feature_cols)}")
    
    del X_train, X_test, y_train, y_test
    gc.collect()
    
    return data

training_data = prep_training_data(
    ml_data,
    sequence_length=16,
    test_size=0.1,
    use_bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
)

# === Noise / signal diagnostics ===
import numpy as np, pandas as pd

r_train = training_data['raw']['y_train']
print("\n=== Return Noise Diagnostics (Train) ===")
# Basic stats
print(f"Samples: {len(r_train)}")
print(f"Mean: {r_train.mean():.6f}  Std: {r_train.std():.6f}  Mean|r|: {np.mean(np.abs(r_train)):.6f}")

# Directional balance
sign_r = np.sign(r_train)
pos_ratio = (sign_r > 0).mean()
print(f"Positive return ratio: {pos_ratio:.3f}")

# Autocorrelation lags 1..5
for lag in range(1, 6):
    ac = np.corrcoef(r_train[:-lag], r_train[lag:])[0, 1]
    print(f"Lag {lag} autocorr: {ac:.5f}")

# Aggregated horizons (e.g., 4×15m = 1h, 8×15m = 2h)
r_series = pd.Series(r_train)
agg_map = {'1h(4)':4, '2h(8)':8, '4h(16)':16}
for label, w in agg_map.items():
    agg = r_series.rolling(w).sum().dropna()
    print(f"{label} std: {agg.std():.6f}  mean|agg|: {np.mean(np.abs(agg)):.6f}")

# Variance ratio (compare sqrt scaling)
import math
base_std = r_series.std()
for label, w in agg_map.items():
    agg_std = r_series.rolling(w).sum().dropna().std()
    vr = agg_std / (base_std * math.sqrt(w))
    print(f"{label} variance ratio vs random walk: {vr:.3f}")

# Baseline directional accuracies (test set)
r_test = training_data['raw']['y_test']
sign_test = np.sign(r_test)
maj_class = 1 if pos_ratio >= 0.5 else -1
baseline_majority = (sign_test == maj_class).mean()*100
baseline_zero = (np.sign(np.zeros_like(r_test)) == sign_test).mean()*100
prev_sign_pred = np.sign(np.r_[0, r_test[:-1]])
baseline_prev = (prev_sign_pred == sign_test).mean()*100
print("\n=== Directional Baselines (Test) ===")
print(f"Majority class: {baseline_majority:.2f}%")
print(f"Always zero:    {baseline_zero:.2f}%")
print(f"Prev sign:      {baseline_prev:.2f}%")
print("====================================================")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 40)

In [ ]:

# PYTORCH TRAINING FUNCTIONS AND EXECUTION
from torch import autocast

def train_pytorch_model(model, data, epochs, model_type, batch_size, lr):
    """
    Train PyTorch model and track directional accuracy (Supports GPU)
    
    Args:
        model: PyTorch model
        data: Prepared data dictionary  
        model_type: 'regular' or 'sequence'
        epochs: Number of training epochs
        batch_size: Batch size for training
        lr: Learning rate
        use_mixed_precision: Use mixed precision, will automatically be set to true if cuda is detected
    """ 
    # --- BF16 detection ---
    use_bf16 = data.get('use_bf16', False)
    tensor_dtype = data.get('tensor_dtype', torch.float32)
    
    model = model.to(device).to(dtype=tensor_dtype)
        
    # torch.compile can be flaky with cuDNN LSTM; skip for sequence models
    if model_type != 'sequence':
        try:
            model = torch.compile(model, mode="default", dynamic=True)
        except Exception as e:
            print("compile skipped:", e)
    
    # Build datasets on CPU
    if model_type == 'sequence':
        seq_len = data.get('sequence_length', 16)
        Xtr = data['raw']['X_train']; ytr = data['raw']['y_train']; ttr = data['raw']['t_train']
        Xte = data['raw']['X_test'];  yte = data['raw']['y_test'];  tte = data['raw']['t_test']

        train_dataset = SequenceDataset(Xtr, ytr, ttr, seq_len, to_dtype=tensor_dtype)
        test_dataset  = SequenceDataset(Xte, yte, tte, seq_len, to_dtype=tensor_dtype)
    else:
        Xtr = torch.from_numpy(data['raw']['X_train'])
        ytr = torch.from_numpy(data['raw']['y_train'].astype(np.int64, copy=False))  # CrossEntropy needs long
        Xte = torch.from_numpy(data['raw']['X_test'])
        yte = torch.from_numpy(data['raw']['y_test'].astype(np.int64, copy=False))

        train_dataset = TensorDataset(Xtr, ytr)
        test_dataset  = TensorDataset(Xte, yte)
    
    # DataLoaders: pin memory + workers to stream batches
    if device.type == 'cuda':
        num_workers = max(1, os.cpu_count() // 2)
        train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=(model_type!='sequence'),
            drop_last=True, pin_memory=(device.type=='cuda'), num_workers=num_workers,
            persistent_workers=(num_workers>0), prefetch_factor=2
        )
        test_loader = DataLoader(
            test_dataset, batch_size=batch_size, shuffle=False,
            pin_memory=(device.type=='cuda'), num_workers=num_workers,
            persistent_workers=(num_workers>0), prefetch_factor=2
        )
        
        train_loader = CUDAPrefetchLoader(train_loader, device, tensor_dtype, prefetch=4)
        test_loader  = CUDAPrefetchLoader(test_loader,  device, tensor_dtype, prefetch=2)
    else:
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=(model_type!='sequence'), drop_last=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.15, # def/2
        div_factor=25, # def
        final_div_factor=1e4, # def
        anneal_strategy='cos' # def
    )    
    # Training metrics
    train_losses, test_losses, directional_accuracies = [], [], []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            
            batch_X = batch_X.to(device, dtype=tensor_dtype)
            batch_y = batch_y.to(device, dtype=torch.int64)
            
            if use_bf16:
                # BF16: autocast + regular backward (no GradScaler)
                with autocast(device_type='cuda', dtype=tensor_dtype):
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y.long())
                loss.backward()
            else:
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y.long())
                loss.backward()
                
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_loss += float(loss.detach().cpu())
            
        # Eval
        model.eval()
        test_loss = 0.0
        correct_predictions = 0
        total_predictions = 0
        
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.contiguous().to(device, dtype=tensor_dtype, non_blocking=True)
                batch_y = batch_y.to(device, dtype=torch.int64, non_blocking=True)
                if use_bf16:
                    with autocast(device_type='cuda', dtype=tensor_dtype):
                        outputs = model(batch_X)
                else:
                    outputs = model(batch_X)

                test_loss += float(loss.detach().cpu())
         
                # Calculate accuracy
                _, predicted = torch.max(outputs, 1)
                total_predictions += batch_y.size(0)
                correct_predictions += (predicted == batch_y.long()).sum().item()
        
        accuracy = 100 * correct_predictions / total_predictions
        avg_train_loss = train_loss / len(train_loader)
        avg_test_loss = test_loss / len(test_loader)
        
        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)
        directional_accuracies.append(accuracy)
        
        print(f'Epoch {epoch:3d}: Train Loss: {avg_train_loss:.6f}, Test Loss: {avg_test_loss:.6f}, Accuracy: {accuracy:.1f}%')
    # Final evaluation
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.contiguous()
            batch_X = batch_X.to(device, dtype=tensor_dtype, non_blocking=True)
            batch_y = batch_y.to(device, dtype=torch.int64, non_blocking=True)
            if use_bf16:
                with autocast(device_type='cuda', dtype=tensor_dtype):
                    outputs = model(batch_X)
            else:
                outputs = model(batch_X)

            predictions.extend(predicted.cpu().numpy().tolist())
            actuals.extend(batch_y.cpu().numpy().tolist())

    # --- NEW: REVERT TRANSFORMATIONS BEFORE CALCULATING METRICS ---
    predictions_np = np.array(predictions)
    actuals_np = np.array(actuals)
    
    # Classification accuracy
    accuracy = np.mean(predictions_np == actuals_np) * 100
    
    # For classification, we need different metrics
    from sklearn.metrics import classification_report, confusion_matrix
    from sklearn.metrics import accuracy_score, f1_score
    
    accuracy_final = accuracy_score(actuals_np, predictions_np) * 100
    f1_weighted = f1_score(actuals_np, predictions_np, average='weighted')
    
    print(f"📊 Final Metrics - Accuracy: {accuracy_final:.2f}%, F1-Score: {f1_weighted:.4f}")
    print("\n📋 Classification Report:")
    print(classification_report(actuals_np, predictions_np, target_names=['Down', 'Neutral', 'Up']))
    
    if USE_FULL_DATASET:
            torch.save(model.state_dict(), f'models_full_dataset/{model.__class__.__name__.lower()}_f.pth')
    else:
        torch.save(model.state_dict(), f'models_limited/{model.__class__.__name__.lower()}_l.pth')
    
    return {
        'model': model,
        'train_losses': train_losses,
        'test_losses': test_losses,
        'directional_accuracies': directional_accuracies,
        'predictions': predictions,           # ✅ Class predictions
        'actuals': actuals_np.tolist(),      # ✅ Class labels
        'metrics': {'accuracy': accuracy_final, 'f1_score': f1_weighted}
    }

In [ ]:
# TRAIN ALL MODELS
print(f"🔥 TRAINING NEURAL NETWORKS ON {device}")
print("=" * 50)

if USE_FULL_DATASET:
    global_epochs = 8
    global_batch_size = 256     # Larger batches for efficiency
    global_lr = 0.005          # Lower learning rate for stability
else:
    global_epochs = 20
    global_batch_size = 64
    global_lr = 0.005
    
results = {}

# 2. Train LSTM Network
print("\n2️⃣ TRAINING LSTM NETWORK")
print("-" * 25)
lstm_model = LSTM(input_features=len(training_data['feature_names'])).to(device)
results['LSTM'] = train_pytorch_model(
    lstm_model, training_data, model_type='sequence', epochs=global_epochs,
    batch_size=512 if USE_FULL_DATASET else 64,
    lr=global_lr
)

# 1. Train Feed-Forward Neural Network
print("\n1️⃣ TRAINING FEED-FORWARD NEURAL NETWORK")
print("-" * 45)
ff_model = FeedForward(input_features=len(training_data['feature_names'])).to(device)
results['FeedForward'] = train_pytorch_model(
    ff_model, training_data, model_type='regular', epochs=global_epochs,
    batch_size=4096 if USE_FULL_DATASET else 64,
    lr=global_lr
)


# 3. Train Hybrid Network
print("\n3️⃣ TRAINING HYBRID CNN-LSTM NETWORK") 
print("-" * 35)
hybrid_model = Hybrid(input_features=len(training_data['feature_names'])).to(device)
results['Hybrid'] = train_pytorch_model(
    hybrid_model, training_data, model_type='sequence', epochs=global_epochs,
    batch_size=512 if USE_FULL_DATASET else 64,
    lr=global_lr
)
print("\n✅ PyTorch GPU Training Complete!")
print("=" * 50)

In [ ]:
## HELPER METHODS

# Calculate enhanced metrics for all models
def calculate_enhanced_metrics(results, training_data):
    """Calculate trading-focused metrics for model evaluation"""
    enhanced_results = {}
    
    log_transform_used = training_data['log_transform_target']
    outlier_threshold = 10 if log_transform_used else 50
    
    for name, result in results.items():
        actuals = np.array(result['actuals'])
        predictions = np.array(result['predictions'])
        
        # Convert to percentage returns
        actual_returns = actuals * 100
        predicted_returns = predictions * 100
        
        # Filter outliers
        mask = (np.abs(actual_returns) < outlier_threshold) & (np.abs(predicted_returns) < outlier_threshold)
        actual_clean = actual_returns[mask]
        predicted_clean = predicted_returns[mask]

        # Trading-focused metrics
        direction_correct = np.sign(actual_clean) == np.sign(predicted_clean)
        directional_accuracy = np.mean(direction_correct) * 100
        
        # Hit rate for significant moves (>1% moves)
        significant_moves = np.abs(actual_clean) > 1.0
        if np.sum(significant_moves) > 0:
            hit_rate_significant = np.mean(direction_correct[significant_moves]) * 100
        else:
            hit_rate_significant = 0
        
        # Return prediction accuracy
        return_mae = np.mean(np.abs(actual_clean - predicted_clean))
        return_rmse = np.sqrt(np.mean((actual_clean - predicted_clean)**2))
        
        # Volatility matching
        actual_vol = np.std(actual_clean)
        predicted_vol = np.std(predicted_clean)
        vol_ratio = predicted_vol / actual_vol if actual_vol > 0 else 0
        
        # Sharpe-like ratio for predictions (mean return / volatility)
        mean_predicted_return = np.mean(predicted_clean)
        sharpe_like = mean_predicted_return / predicted_vol if predicted_vol > 0 else 0
        
        # Correlation
        correlation = np.corrcoef(actual_clean, predicted_clean)[0, 1]
        
        enhanced_results[name] = {
            'directional_accuracy': directional_accuracy,
            'hit_rate_significant': hit_rate_significant,
            'return_mae': return_mae,
            'return_rmse': return_rmse,
            'correlation': correlation,
            'actual_vol': actual_vol,
            'predicted_vol': predicted_vol,
            'vol_ratio': vol_ratio,
            'sharpe_like': sharpe_like,
            'actual_clean': actual_clean,
            'predicted_clean': predicted_clean,
            'samples': len(actual_clean)
        }
    
    return enhanced_results

# Find best model based on composite score
def calculate_composite_score(metrics):
    """Calculate composite score emphasizing trading performance"""
    # Normalize metrics (higher is better)
    dir_acc_norm = metrics['directional_accuracy'] / 100  # 0-1 scale
    corr_norm = (metrics['correlation'] + 1) / 2  # -1,1 to 0,1 scale
    vol_ratio_norm = 1 - abs(1 - metrics['vol_ratio'])  # Penalty for being far from 1
    
    # Penalty for high MAE (lower is better)
    mae_penalty = 1 / (1 + metrics['return_mae'])  # Higher MAE = lower score
    
    # Composite score (weights can be adjusted)
    composite = (
        0.4 * dir_acc_norm +      # 40% directional accuracy
        0.3 * corr_norm +         # 30% correlation
        0.2 * mae_penalty +       # 20% MAE penalty
        0.1 * vol_ratio_norm      # 10% volatility matching
    )
    
    return composite

In [ ]:
# IMPROVED PYTORCH RESULTS VISUALIZATION - TRUE BLACK DARK MODE

import matplotlib.pyplot as plt
import matplotlib

# Set dark theme for plots with TRUE BLACK backgrounds
plt.style.use('dark_background')
matplotlib.rcParams.update({
    'figure.facecolor': '#000000',      # Pure black
    'axes.facecolor': '#000000',        # Pure black
    'savefig.facecolor': '#000000',     # Pure black
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': 'white',
    'grid.color': '#404040',
    'axes.spines.bottom': True,
    'axes.spines.top': True,
    'axes.spines.right': True,
    'axes.spines.left': True
})

# Bright colors that pop against pure black
dark_colors = {
    'blue': '#00E5FF', 'purple': '#C77DFF', 'orange': '#FF9F40',
    'red': '#FF5555', 'coral': '#FF8A80', 'teal': '#1DE9B6',
    'yellow': '#FFEB3B', 'green': '#69F0AE'
}

# Define models dictionary and calculate metrics
models = {name: result['model'] for name, result in results.items()}
enhanced_metrics = calculate_enhanced_metrics(results, training_data)
composite_scores = {name: calculate_composite_score(metrics) for name, metrics in enhanced_metrics.items()}
best_model_enhanced = max(composite_scores, key=lambda k: composite_scores[k])

# Create visualization with accuracy plot
fig = plt.figure(figsize=(30, 22), facecolor='#000000')
gs = fig.add_gridspec(5, 4, hspace=0.35, wspace=0.3)

fig.suptitle(f'Performance Analysis: {global_epochs} epochs, {STOCK_LIMIT} stocks, {len(training_data["feature_names"])} params, {date.today()}', 
             fontsize=16, fontweight='bold', color='white')
fig.text(0.5, 0.96, f"Log transform: {training_data['log_transform_target']}", ha='center', fontsize=13, color='white')

models_list = list(enhanced_metrics.keys())

def create_bar_chart(ax, x_data, y_data_list, labels, colors, title, ylabel, loc, add_random_line=False):
    """Helper function to create consistent bar charts"""
    ax.set_facecolor('#000000')
    x = np.arange(len(x_data))
    width = 0.25 if len(y_data_list) == 3 else 0.4
    
    bars_list = []
    for i, (y_data, label, color) in enumerate(zip(y_data_list, labels, colors)):
        offset = (i - len(y_data_list)//2) * width
        bars = ax.bar(x + offset, y_data, width, label=label, alpha=0.9, color=color)
        bars_list.append(bars)
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + (0.05 if height < 10 else 1),
                    f'{height:.1f}' if height > 1 else f'{height:.2f}', 
                    ha='center', va='bottom', fontsize=9, color='white', fontweight='bold')
    
    if add_random_line:
        ax.axhline(y=50, color='#FF5555', linestyle='--', alpha=0.8, linewidth=2)
    
    ax.set_title(title, fontweight='bold', fontsize=14, color='white')
    ax.set_xlabel('Models', color='white')
    ax.set_ylabel(ylabel, color='white')
    ax.set_ylim(0, np.max(y_data_list) + 0.25 * np.max(y_data_list))
    ax.set_xticks(x)
    ax.set_xticklabels(x_data, color='white')
    ax.legend(framealpha=0.9, facecolor='#1a1a1a', edgecolor='white', loc=loc)
    ax.grid(True, alpha=0.3, color='#505050')
    
    return bars_list

# 1. Trading Performance Dashboard (Top row, spans 2 columns)
ax1 = fig.add_subplot(gs[0, :2])
dir_acc = [enhanced_metrics[m]['directional_accuracy'] for m in models_list]
hit_rate_sig = [enhanced_metrics[m]['hit_rate_significant'] for m in models_list]
correlation = [enhanced_metrics[m]['correlation'] * 100 for m in models_list]

create_bar_chart(ax1, models_list, 
                [dir_acc, hit_rate_sig, correlation],
                ['Directional Accuracy (%)', 'Hit Rate >1% moves (%)', 'Correlation (%)'],
                [dark_colors['blue'], dark_colors['purple'], dark_colors['orange']],
                'Trading Performance Metrics', 'Performance (%)', 'lower left', add_random_line=True)

# 2. Return Prediction Quality (Top row, right)
ax2 = fig.add_subplot(gs[0, 2:])
mae_values = [enhanced_metrics[m]['return_mae'] for m in models_list]
rmse_values = [enhanced_metrics[m]['return_rmse'] for m in models_list]

create_bar_chart(ax2, models_list,
                [mae_values, rmse_values],
                ['MAE (pp)', 'RMSE (pp)'],
                [dark_colors['red'], dark_colors['coral']],
                'Return Prediction Errors', 'Error (Percentage Points)', 'lower left')

# 3. NEW: Directional Accuracy Over Epochs (Second row, spans full width)
ax3 = fig.add_subplot(gs[1, :])
ax3.set_facecolor('#000000')

colors_list = [dark_colors['blue'], dark_colors['purple'], dark_colors['orange']]
for i, (model_name, result) in enumerate(results.items()):
    if 'directional_accuracies' in result:
        epochs = range(len(result['directional_accuracies']))
        accuracies = result['directional_accuracies']
        
        ax3.plot(epochs, accuracies, label=f'{model_name}', 
                color=colors_list[i % len(colors_list)], linewidth=3, alpha=0.9, marker='o', markersize=4)
        
        # Add final accuracy value as text
        final_acc = accuracies[-1]
        ax3.text(len(epochs)-1, final_acc, f'{final_acc:.1f}%', 
                color=colors_list[i % len(colors_list)], fontweight='bold', fontsize=10,
                ha='left', va='center')

ax3.set_title('Directional Accuracy Over Training Epochs', fontweight='bold', fontsize=16, color='white')
ax3.set_xlabel('Epoch', color='white', fontsize=12)
ax3.set_ylabel('Directional Accuracy (%)', color='white', fontsize=12)
ax3.legend(framealpha=0.9, facecolor='#1a1a1a', edgecolor='white', fontsize=12)
ax3.grid(True, alpha=0.3, color='#505050')
ax3.axhline(y=50, color='#FF5555', linestyle='--', alpha=0.8, linewidth=2, label='Random Baseline (50%)')
ax3.axhline(y=60, color='#00FF00', linestyle=':', alpha=0.6, linewidth=1, label='Good Performance (60%)')

# Set y-axis limits for better visualization
if any('directional_accuracies' in result for result in results.values()):
    max_acc = max([max(result['directional_accuracies']) for result in results.values() if 'directional_accuracies' in result])
    ax3.set_ylim(45, max_acc + 5)

# 4. Best Model Detailed Analysis (Third row, left)
ax4 = fig.add_subplot(gs[2, :2])
ax4.set_facecolor('#000000')
if best_model_enhanced in enhanced_metrics:
    best_metrics = enhanced_metrics[best_model_enhanced]
    actual_clean = best_metrics['actual_clean']
    predicted_clean = best_metrics['predicted_clean']
    
    # Color code: green for correct direction, red for incorrect
    colors = [dark_colors['green'] if np.sign(a) == np.sign(p) else dark_colors['red'] 
              for a, p in zip(actual_clean, predicted_clean)]
    
    ax4.scatter(actual_clean, predicted_clean, c=colors, alpha=0.8, s=30, edgecolors='white', linewidth=0.7)
    
    # Perfect prediction line
    min_val, max_val = min(min(actual_clean), min(predicted_clean)), max(max(actual_clean), max(predicted_clean))
    ax4.plot([min_val, max_val], [min_val, max_val], color='white', linestyle='--', alpha=0.9, linewidth=3)
    
    ax4.set_title(f'{best_model_enhanced}: Actual vs Predicted Returns', fontweight='bold', fontsize=14, color='white')
    ax4.set_xlabel('Actual Returns (%)', color='white')
    ax4.set_ylabel('Predicted Returns (%)', color='white')
    ax4.grid(True, alpha=0.3, color='#505050')
    ax4.axhline(y=0, color='#808080', linestyle='-', alpha=0.6)
    ax4.axvline(x=0, color='#808080', linestyle='-', alpha=0.6)

# 5. Volatility Analysis (Third row, right)
ax5 = fig.add_subplot(gs[2, 2:])
actual_vols = [enhanced_metrics[m]['actual_vol'] for m in models_list]
predicted_vols = [enhanced_metrics[m]['predicted_vol'] for m in models_list]

create_bar_chart(ax5, models_list,
                [actual_vols, predicted_vols],
                ['Actual Volatility', 'Predicted Volatility'],
                [dark_colors['teal'], dark_colors['yellow']],
                'Volatility Matching', 'Volatility (%)', 'lower right')

# Add ratio labels for volatility
for i, (actual, predicted) in enumerate(zip(actual_vols, predicted_vols)):
    ratio = predicted / actual if actual > 0 else 0
    ax5.text(i, max(actual, predicted) + 0.1, f'Ratio: {ratio:.2f}', 
             ha='center', va='bottom', fontsize=9, fontweight='bold', color='white')

# 6. Model Comparison Radar Chart (Fourth row, left)
ax6 = fig.add_subplot(gs[3, :2], projection='polar')
ax6.set_facecolor('#000000')

metrics_names = ['Directional\nAccuracy', 'Correlation', 'Low MAE', 'Vol Match', 'Composite\nScore']
angles = np.linspace(0, 2 * np.pi, len(metrics_names), endpoint=False).tolist() + [0]

for i, model in enumerate(models_list):
    if model in enhanced_metrics:
        m = enhanced_metrics[model]
        values = [
            m['directional_accuracy'] / 100,
            (m['correlation'] + 1) / 2,
            1 / (1 + m['return_mae']),
            1 - abs(1 - m['vol_ratio']),
            composite_scores[model]
        ] + [m['directional_accuracy'] / 100]  # Close the circle
        
        ax6.plot(angles, values, 'o-', linewidth=4, label=model, alpha=0.9, color=colors_list[i % len(colors_list)])
        ax6.fill(angles, values, alpha=0.2, color=colors_list[i % len(colors_list)])

ax6.set_xticks(angles[:-1])
ax6.set_xticklabels(metrics_names, color='white')
ax6.set_ylim(0, 1)
ax6.set_title('Multi-Metric Model Comparison', fontweight='bold', fontsize=14, pad=20, color='white')
ax6.legend(loc='upper left', bbox_to_anchor=(1.4, 1.0), framealpha=0.9, facecolor='#1a1a1a', edgecolor='white')
ax6.grid(True, color='#505050')
ax6.tick_params(colors='white')

# 7. Training Loss Evolution (Fourth row, right)
ax7 = fig.add_subplot(gs[3, 2])
ax7.set_facecolor('#000000')

for i, (name, result) in enumerate(results.items()):
    epochs_range = range(len(result['train_losses']))
    ax7.plot(epochs_range, result['train_losses'], label=f'{name} Train', alpha=0.9, linewidth=3, color=colors_list[i % len(colors_list)])

ax7.set_title('Loss Convergence Analysis', fontweight='bold', fontsize=14, color='white')
ax7.set_xlabel('Epoch', color='white')
ax7.set_ylabel('Loss', color='white')
ax7.legend(framealpha=0.9, facecolor='#1a1a1a', edgecolor='white')
ax7.grid(True, alpha=0.3, color='#505050')
ax7.set_yscale('log')

# 8. Validation Loss Evolution (Fourth row, far right)
ax8 = fig.add_subplot(gs[3, 3])
ax8.set_facecolor('#000000')

# Plot only validation loss
for i, (name, result) in enumerate(results.items()):
    epochs_range = range(len(result['test_losses']))
    ax8.plot(epochs_range, result['test_losses'], label=f'{name} Val', linestyle='--', alpha=0.9, linewidth=3, color=colors_list[i % len(colors_list)])

ax8.set_title('Validation Loss Convergence', fontweight='bold', fontsize=14, color='white')
ax8.set_xlabel('Epoch', color='white')
ax8.set_ylabel('Validation Loss', color='white')
ax8.legend(framealpha=0.9, facecolor='#1a1a1a', edgecolor='white')
ax8.grid(True, alpha=0.3, color='#505050')
ax8.set_yscale('log')

# 8. Performance Summary Table (Bottom row, spans full width)
ax9 = fig.add_subplot(gs[4, :])
ax9.axis('tight')
ax9.axis('off')
ax9.set_facecolor('#000000')

headers = ['Model', 'Dir. Acc.', 'Hit Rate >1%', 'Actual range', 'Predicted range', 'MAE (pp)', 'Correlation', 'Vol Ratio', 'Composite Score', 'Samples']
table_data = []

for model in models_list:
    if model in enhanced_metrics:
        m = enhanced_metrics[model]
        table_data.append([
            model,
            f"{m['directional_accuracy']:.1f}%",
            f"{m['hit_rate_significant']:.1f}%",
            f"{np.min(m['actual_clean']):.1f}% - {np.max(m['actual_clean']):.1f}%",
            f"{np.min(m['predicted_clean']):.1f}% - {np.max(m['predicted_clean']):.1f}%",
            f"{m['return_mae']:.2f}",
            f"{m['correlation']:.3f}",
            f"{m['vol_ratio']:.2f}",
            f"{composite_scores[model]:.3f}",
            f"{m['samples']:,}"
        ])

table = ax9.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 3)

# Style the table
for (row, col), cell in table.get_celld().items():
    if row == 0:  # Header row
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#333333')
    else:
        cell.set_text_props(color='white')
        cell.set_facecolor('#000000')
    cell.set_edgecolor('#666666')
    cell.set_linewidth(1)

# Highlight best model row
best_row_idx = models_list.index(best_model_enhanced) + 1
for j in range(len(headers)):
    table[(best_row_idx, j)].set_facecolor('#004d00')

ax9.set_title('Performance Summary Table', fontweight='bold', fontsize=14, color='white')

plt.tight_layout()
plt.subplots_adjust(top=0.93)  # Move plots closer to the title
save_path = f'models_results/{date.today().strftime("%Y-%m-%d_%H-%M")}_{global_epochs}e_{"500" if USE_FULL_DATASET else STOCK_LIMIT}s.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()